# Where to put the ambulances

A walkthrough of the siting analysis: build the coverage geometry, solve set cover
and maximal covering, and look at who ends up waiting.

Everything here calls into `src/`, which is where the models actually live. This
notebook is the narrative; `python -m src.run_analysis` is the reproducible entry
point that writes every table and figure in `results/`.

Run `python -m src.data_prep` first if `data/processed/` is empty.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.coverage import (
    build_coverage_matrix,
    build_time_matrix,
    coverage_summary,
    euclidean_minutes,
    haversine_minutes,
)
from src.equity import attach_demographics, disparity, response_by_group
from src.models import (
    assign_to_nearest,
    solve_max_cover,
    solve_set_cover,
    solve_set_cover_relaxation,
)

PROCESSED = Path.cwd().parent / "data" / "processed"
pd.set_option("display.width", 120)

## 1. Demand

One row per census block group: the centroid, and the number of 911 dispatches
the City of Pittsburgh recorded there between 2015 and 2024. Each block group is
both a demand point and a candidate station site.

In [2]:
points = pd.read_csv(PROCESSED / "demand_points.csv", dtype={"geoid": str, "geoid_2020": str})
demographics = pd.read_csv(PROCESSED / "block_group_demographics.csv", dtype={"geoid": str})

print(f"Demand points: {len(points)}")
print(f"Total calls:   {points['call_count'].sum():,}")
print(f"Busiest block group: {points['call_count'].max():,} calls")
print(f"Median block group:  {points['call_count'].median():,.0f} calls")
points.head()

Demand points: 389
Total calls:   635,235
Busiest block group: 14,481 calls
Median block group:  1,100 calls


,geoid,lon,lat,call_count,geoid_2020
0,420030103001,-79.984817,40.434374,5265,420030103011
1,420030103002,-79.990476,40.436902,2844,420030103021
2,420030103003,-79.983764,40.436901,9300,420030103022
3,420030103004,-79.977290,40.437355,1736,420030103022
4,420030201001,-80.006729,40.439640,12322,420030201001


Demand is uneven but not dominated by a handful of hotspots. The Gini
coefficient across block groups is about 0.49, and the busiest tenth carry
roughly a third of all calls. That matters for the model choice later: if demand
were extremely concentrated, weighting demand points by call volume would change
the answer a great deal. Here it barely does.

In [3]:
calls = np.sort(points["call_count"].to_numpy(dtype=float))
n = len(calls)
gini = 2 * np.sum(np.arange(1, n + 1) * calls) / (n * calls.sum()) - (n + 1) / n

print(f"Gini across block groups: {gini:.3f}")
print(f"Busiest 10% carry:        {100 * calls[-n // 10:].sum() / calls.sum():.1f}% of calls")
print(f"Least busy 50% carry:     {100 * calls[:n // 2].sum() / calls.sum():.1f}% of calls")

Gini across block groups: 0.487
Busiest 10% carry:        36.4% of calls
Least busy 50% carry:     18.4% of calls


## 2. Travel time and the coverage matrix

Travel time is great-circle distance at an assumed 40 km/h. Section 5 shows how
much that constant matters, which is more than the optimisation does.

The coverage matrix `a[i, j]` is 1 when a station at `j` reaches demand point `i`
within the threshold.

In [4]:
THRESHOLD = 10.0

time_matrix = build_time_matrix(points["lat"], points["lon"], haversine_minutes)
coverage = build_coverage_matrix(time_matrix, THRESHOLD)

for key, value in coverage_summary(coverage).items():
    print(f"{key:32s} {value}")

demand_points                    389
candidates                       389
density                          0.5300586171119672
mean_covered_per_candidate       206.19280205655528
min_covered_per_candidate        8
max_covered_per_candidate        309
uncoverable_demand_points        0


A sanity check worth doing once: at the scale of a single city, the choice
between great-circle and flat-earth distance is not a real modelling decision.

In [5]:
flat = build_time_matrix(points["lat"], points["lon"], euclidean_minutes)
print(f"Largest disagreement across the 389 x 389 matrix: {np.abs(time_matrix - flat).max():.4f} minutes")

Largest disagreement across the 389 x 389 matrix: 0.0560 minutes


## 3. Set cover

The cheapest fleet that leaves nobody outside the standard:

$$\min \sum_j y_j \quad \text{s.t.} \quad \sum_j a_{ij} y_j \ge 1 \ \ \forall i, \quad y_j \in \{0, 1\}$$

In [6]:
cover = solve_set_cover(coverage, quiet=True)
lp_bound = solve_set_cover_relaxation(coverage, quiet=True)

print(f"Stations required: {cover.n_facilities}")
print(f"LP relaxation:     {lp_bound:.3f}")
print(f"Integrality gap:   {cover.objective - lp_bound:.3f}")
print(f"Solved in {cover.runtime:.3f}s over {cover.n_variables} binaries and {cover.n_constraints} constraints")

cover_response, _ = assign_to_nearest(time_matrix, cover.selected)
stations = points.iloc[cover.selected][["geoid", "lat", "lon", "call_count"]].reset_index(drop=True)
stations.index = range(1, len(stations) + 1)
stations

Stations required: 4
LP relaxation:     3.500
Integrality gap:   0.500
Solved in 0.029s over 389 binaries and 389 constraints


,geoid,lat,lon,call_count
1,420030804002,40.455130,-79.947900,979
2,420032107001,40.453161,-80.023652,3418
3,420032904004,40.390989,-79.976421,1365
4,420034885001,40.364251,-79.921294,3


In [7]:
weighted = np.average(cover_response, weights=points["call_count"])
print(f"Mean response time:          {cover_response.mean():.2f} min")
print(f"Call-weighted mean:          {weighted:.2f} min")
print(f"Median:                      {np.median(cover_response):.2f} min")
print(f"Worst case:                  {cover_response.max():.2f} min")

Mean response time:          4.46 min
Call-weighted mean:          4.13 min
Median:                      4.55 min
Worst case:                  9.94 min


## 4. Maximal covering, and the station that serves nobody

Set cover has to reach every demand point, including the most awkward one. The
maximal covering problem asks the opposite question: given a fixed budget of `p`
stations, how much can we reach?

$$\max \sum_i v_i z_i \quad \text{s.t.} \quad \sum_j y_j = p, \quad z_i \le \sum_j a_{ij} y_j, \quad y_j, z_i \in \{0,1\}$$

The weight $v_i$ is a policy choice. With $v_i = 1$ every block group counts the
same regardless of how many calls it generates. With $v_i =$ call count, the
model maximises covered call volume instead.

In [8]:
call_weights = points["call_count"].to_numpy(dtype=float)
total_calls = call_weights.sum()

rows = []
for p in range(1, 6):
    by_block = solve_max_cover(coverage, p, weights=None)
    by_call = solve_max_cover(coverage, p, weights=call_weights)
    rows.append({
        "p": p,
        "blocks covered": len(by_block.covered),
        "% blocks": 100 * len(by_block.covered) / len(points),
        "% calls (block objective)": 100 * call_weights[by_block.covered].sum() / total_calls,
        "% calls (call objective)": 100 * by_call.objective / total_calls,
        "same sites": set(by_block.selected) == set(by_call.selected),
    })

pd.DataFrame(rows).set_index("p").round(4)

,blocks covered,% blocks,% calls (block objective),% calls (call objective),same sites
p,,,,,
1,309,79.4344,86.5258,86.6445,False
2,380,97.6864,99.6877,99.6877,True
3,388,99.7429,99.9998,99.9998,True
4,389,100.0000,100.0000,100.0000,True
5,389,100.0000,100.0000,100.0000,True


Three stations reach 99.9998% of call volume. Four are needed only because one
block group sits outside every three-station configuration, and that block group
generated **one call in ten years**.

The two objectives select different sites only at `p = 1`, and even there they
differ by 0.1 percentage points of call volume. On a more spread-out service area
this gap would be much wider, which is why it is worth measuring rather than
assuming.

In [9]:
mclp = solve_max_cover(coverage, 3, weights=None)
missed = sorted(set(range(len(points))) - set(mclp.covered))

print(f"Three stations cover {len(mclp.covered)}/{len(points)} block groups")
print(f"Calls in the block group left out: {points.iloc[missed]['call_count'].sum()}")
points.iloc[missed][["geoid", "lat", "lon", "call_count"]]

Three stations cover 388/389 block groups
Calls in the block group left out: 1


,geoid,lat,lon,call_count
317,420034890013,40.326275,-79.951688,1


## 5. What the answer really rests on

Vary the assumed speed, which is the softest input in the whole pipeline, and the
recommendation moves by two stations. That is a wider swing than anything the
optimisation contributes.

In [10]:
rows = []
for speed in (30.0, 35.0, 40.0, 45.0, 50.0):
    matrix = build_time_matrix(points["lat"], points["lon"], haversine_minutes, speed_kmh=speed)
    solution = solve_set_cover(build_coverage_matrix(matrix, THRESHOLD), quiet=True)
    response, _ = assign_to_nearest(matrix, solution.selected)
    rows.append({
        "speed (km/h)": speed,
        "10-min reach (km)": speed / 60 * THRESHOLD,
        "stations": solution.n_facilities,
        "mean response (min)": response.mean(),
    })

pd.DataFrame(rows).set_index("speed (km/h)").round(2)

,10-min reach (km),stations,mean response (min)
speed (km/h),,,
30.0,5.00,5,4.98
35.0,5.83,4,4.64
40.0,6.67,4,4.46
45.0,7.50,3,4.53
50.0,8.33,3,4.35


In [11]:
rows = []
for threshold in (5, 6, 7, 8, 9, 10, 12, 15, 20):
    cov = build_coverage_matrix(time_matrix, threshold)
    if coverage_summary(cov)["uncoverable_demand_points"]:
        continue
    solution = solve_set_cover(cov, quiet=True)
    rows.append({
        "threshold (min)": threshold,
        "stations": solution.n_facilities,
        "LP bound": solve_set_cover_relaxation(cov, quiet=True),
        "integrality gap": solution.objective - solve_set_cover_relaxation(cov, quiet=True),
    })

sensitivity = pd.DataFrame(rows).set_index("threshold (min)")
sensitivity.round(3)

,stations,LP bound,integrality gap
threshold (min),,,
5,10,9.125,0.875
6,7,7.000,0.000
7,6,6.000,0.000
8,4,4.000,0.000
9,4,4.000,0.000
10,4,3.500,0.500
12,3,3.000,0.000
15,2,2.000,0.000
20,1,1.000,0.000


Eight, nine and ten minutes all cost four stations. The cost only starts climbing
below eight minutes. A debate over "10 versus 9" is not a real debate; a debate
over "8 versus 6" is a doubling of capital cost.

The LP relaxation is integral at most thresholds and the largest gap anywhere is
0.88. Set cover is NP-hard in general, but this instance is easy, and saying so
is more useful than implying otherwise.

## 6. Equity

Neither model knows anything about demographics. Any disparity here is a
consequence of where demand and geography put the stations, not of anything the
objective was told to optimise.

In [12]:
mclp_response, _ = assign_to_nearest(time_matrix, mclp.selected)

cover_equity = response_by_group(
    attach_demographics(points, demographics, cover_response), THRESHOLD, 5.0
)
mclp_equity = response_by_group(
    attach_demographics(points, demographics, mclp_response), THRESHOLD, 5.0
)

print("Set cover, 4 stations:")
display(cover_equity.round(2))
print("Maximal covering, 3 stations:")
display(mclp_equity.round(2))

print(f"Call-weighted spread, set cover: {disparity(cover_equity):.2f} min")
print(f"Call-weighted spread, MCLP:      {disparity(mclp_equity):.2f} min")

Set cover, 4 stations:


,Group,Block groups,Calls,Mean minutes,Call-weighted mean minutes,Median minutes,P95 minutes,% calls within 10 min,% calls within 5 min
0,Majority White,273,422300,4.35,3.89,4.51,7.51,100.0,70.43
1,Majority Black,72,142006,5.00,4.84,4.89,8.80,100.0,58.58
2,Majority Asian,1,839,4.74,4.74,4.74,4.74,100.0,100.00
3,Other or mixed,33,48408,3.88,3.81,4.16,6.07,100.0,78.53
4,No population,10,21682,5.28,4.91,5.27,7.96,100.0,56.16


Maximal covering, 3 stations:


,Group,Block groups,Calls,Mean minutes,Call-weighted mean minutes,Median minutes,P95 minutes,% calls within 10 min,% calls within 5 min
0,Majority White,273,422300,5.63,5.38,5.71,8.53,100.0,42.84
1,Majority Black,72,142006,3.78,3.93,3.35,7.50,100.0,61.80
2,Majority Asian,1,839,4.29,4.29,4.29,4.29,100.0,100.00
3,Other or mixed,33,48408,5.04,4.88,5.49,6.99,100.0,39.00
4,No population,10,21682,5.19,4.13,4.76,8.27,100.0,58.73


Call-weighted spread, set cover: 1.04 min
Call-weighted spread, MCLP:      1.46 min


The two models trade the disparity in opposite directions. Under set cover,
majority-Black block groups wait about a minute longer than majority-White ones.
Under maximal covering the ordering inverts, because three stations concentrate
near the dense, high-call-rate core while set cover's fourth station reaches the
sparse periphery.

Two caveats. The majority-Asian row is a single block group and is not a
statement about a population. And this measures response-time equity only;
per-capita call burden differs sharply across groups and is a separate question.

## 7. A data problem that changes the equity numbers

The dispatch GEOIDs are 2010-vintage; the shapefile and the demographic extract
are 2020-vintage. Joining on the GEOID string, which is the obvious move, drops
87 of 389 block groups carrying 26.4% of all calls, and it does so silently.

`src/data_prep.py` joins on geography instead, locating each 2010 centroid inside
the 2020 block group that actually contains it.

In [13]:
naive = points["geoid"].isin(set(demographics["geoid"]))
spatial = points["geoid_2020"].isin(set(demographics["geoid"]))
total = points["call_count"].sum()

print(f"String join:   {naive.sum():>3}/{len(points)} points, "
      f"{100 * points.loc[naive, 'call_count'].sum() / total:.1f}% of calls")
print(f"Spatial join:  {spatial.sum():>3}/{len(points)} points, "
      f"{100 * points.loc[spatial, 'call_count'].sum() / total:.1f}% of calls")
print(f"Calls that the string join loses: {points.loc[~naive, 'call_count'].sum():,}")

String join:   302/389 points, 73.6% of calls
Spatial join:  389/389 points, 100.0% of calls
Calls that the string join loses: 167,467


## 8. Maps and full outputs

`python -m src.run_analysis` writes every table and figure in `results/`, plus two
self-contained interactive maps with layered choropleths for response time, call
volume, calls per capita, population and racial composition:

- `results/maps/set_cover_solution.html`
- `results/maps/mclp_solution.html`